# 4. Unsupported direct evidence

The saved diagnostic says direct coverage did not establish all proposed components. The original rejected response was not retained. The example below deliberately reconstructs that condition using a saved target and passage ID; its claim is illustrative, not a quotation or scientific finding.

Run cells from top to bottom. Everything runs offline; no API key or model request is used.

In [ ]:
from pathlib import Path
import sys
import json
from pprint import pprint

# Works when opened from the repository, notebooks/, or its parent directory.
root = next(p for base in [Path.cwd(), *Path.cwd().parents]
            for p in [base, base / "research_finder"]
            if (p / "novelty/comparison_workflow.py").exists())
sys.path.insert(0, str(root))
# Check the selected kernel before importing project modules.
import importlib.util
missing = [name for name in ("openai", "pydantic", "pymupdf")
           if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Kernel {sys.executable} is missing: {', '.join(missing)}. "
        f"In VS Code choose Select Kernel > Python Environments > {root / '.venv/bin/python'}, "
        "then restart the kernel and Run All."
    )
from notebooks.example_helpers import load_example, expect_failure
from extraction.research_extract import _parse_json
from novelty.comparison_records import EvidenceReport, EvidenceRecord, EvidenceReview
from novelty.comparison_schemas import Passage
from novelty.comparison_workflow import validate_evidence_report, validate_record, eligible, scope_citation_gaps
from novelty.schemas import TargetSignature


In [ ]:
example = load_example(root, "04_unsupported_direct_evidence")
pprint(example["provenance"])

In [ ]:
print("Actual recorded failure:", example["recorded_error"])
target = TargetSignature.model_validate(example["target"])
passage = Passage.model_validate(example["passage"])
print("Proposed intervention:")
pprint([f.model_dump() for f in target.intervention])

## Construct a schema-valid but unjustified direct-coverage declaration
The example supplies a result claim but no mapping from the prior work to the proposed intervention, comparator, regime and effect.

In [ ]:
record = EvidenceRecord.model_validate({
    "claims": [{"claim_id": "demo-result", "text": "Illustrative result only; not asserted to be supported by this passage.", "kind": "result", "passage_ids": [passage.passage_id]}],
    "relationships": [{"target_id": target.target_id, "intervention": [], "comparator": [], "conditions": [], "outcome": [], "conclusion": ["demo-result"], "coverage": "direct_empirical", "result": "supports_prediction", "inference": None, "proposal_matches": [], "explanation": "Deliberately incomplete teaching example."}]
})
error = expect_failure(lambda: validate_record(record, [target], [passage], require_alignment=True), "requires all proposed")

The schema accepts the structure, but the relationship check rejects the unsupported declaration. Related background evidence cannot automatically establish that the proposed experiment was tested. This deterministic check verifies mapping completeness; independent evidence review must still judge whether each mapping is scientifically supported.